## Video Scene Analysis — Repository Integration Demo

This notebook exercises the full `video_analysis_poc` module stack against a new dashcam video.

**Modules used:**

- `cv_preprocessing.CameraProfileDetector` — optical flow camera classification
- `cv_preprocessing.KeyFrameExtractor` — still frame extraction at critical timestamp
- `knowledge_graph.CollisionKnowledgeGraph` — domain expertise graph (from defaults)
- `knowledge_graph.KGRetriever` — scored pattern + regulation retrieval
- `knowledge_graph.format_rag_context / format_verification_misid_check` — prompt formatting
- `prompt_composer.PromptComposer` — composable prompt assembly (system task, verification)
- `schema.video.*` — Pydantic output models

**Pipeline:** Video &rarr; Camera Detection &rarr; KG Retrieval &rarr; Prompt Composition &rarr; LLM Pass 1 (CoT) &rarr; Verification Pass 2 &rarr; Pydantic Output


### 1. Setup

In [ ]:
import litellm
import json, base64, logging, time, sys
from pathlib import Path
from IPython.display import display, JSON, Image as IPImage

logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(levelname)s — %(message)s")
logger = logging.getLogger(__name__)

# Ensure project root is importable
sys.path.insert(0, str(Path(".").resolve().parent))

In [ ]:
# ── LiteLLM credentials ─────────────────────────────────────────────
# Replace with your own credentials or load from environment variables.

LITELLM_API_KEY = ""
LITELLM_BASE_URL = ""

litellm.api_key = LITELLM_API_KEY
litellm.api_base = LITELLM_BASE_URL
litellm.suppress_debug_info = True

# Model identifier — routed through LiteLLM to the Gemini 2.5 Pro deployment
MODEL_NAME = ""

print(f"LiteLLM configured — base: {LITELLM_BASE_URL}")
print(f"Model: {MODEL_NAME}")

### 2. Import Repository Modules

In [ ]:
# CV Pre-processing
from video_analysis_poc.video_llm.cv_preprocessing import (
    CameraProfileDetector, KeyFrameExtractor,
)
# Knowledge Graph
from video_analysis_poc.video_llm.knowledge_graph import (
    CollisionKnowledgeGraph, KGRetriever, format_rag_context,
)
from video_analysis_poc.video_llm.knowledge_graph.formatter import format_verification_misid_check

# Prompt Composer
from video_analysis_poc.video_llm.prompt_composer import PromptComposer

# Pydantic Schemas
from video_analysis_poc.video_llm.schema.video import (
    VideoObjectObservation, CrashInvolvedParty, WitnessedCrashDetail,
    SceneEvent, EnvironmentContext, ReasoningTrace, VideoSceneAnalysisOutput,
)
print("All repository modules imported.")

### 3. Video & Camera Auto-Detection

In [ ]:
VIDEO_PATH = Path("")  # Update to your video path
CAMERA_TYPE = "dashcam_front"
VIDEO_MIME_TYPES = {"mp4": "video/mp4", "avi": "video/x-msvideo",
                    "mov": "video/quicktime", "mkv": "video/x-matroska"}

assert VIDEO_PATH.exists(), f"Video not found: {VIDEO_PATH}"
size_mb = VIDEO_PATH.stat().st_size / (1024 * 1024)
print(f"Video: {VIDEO_PATH.name} ({size_mb:.2f} MB)")

# Camera auto-detection via optical flow
camera_profile = CameraProfileDetector(sample_seconds=5).detect(VIDEO_PATH)
print(f"Camera: {camera_profile.motion_type.value} (stability: {camera_profile.avg_stability:.3f})")
print(f"Context: {camera_profile.prompt_context}")

### 4. Key Frame Extraction

In [ ]:
# Timestamps: approach (0-10s), junction entry (10-20s), stationary period / incident zone (20-45s), aftermath (45-55s)
KEY_TIMESTAMPS = [0, 5, 8, 10, 12, 14, 16, 18, 20, 22, 25, 28, 30, 35, 40, 45, 50, 55]

extractor = KeyFrameExtractor(max_dimension=768, jpeg_quality=85)
key_frames = extractor.extract(VIDEO_PATH, KEY_TIMESTAMPS)
print(f"Extracted {len(key_frames)} key frames at: {[kf.timestamp_seconds for kf in key_frames]}")

In [ ]:
# Quick visual check of a few key frames
for kf in key_frames:
    if kf.timestamp_seconds in [0, 10, 20, 40]:
        display(IPImage(data=base64.b64decode(kf.base64_jpeg), format="jpeg", width=480))
        print(f"  ↑ {kf.timestamp_seconds:.0f}s")

### 5. Knowledge Graph - Build, Retrieve, Format

In [ ]:
# Build KG from embedded defaults
kg = CollisionKnowledgeGraph.from_defaults()
print(f"Knowledge graph: {kg.num_nodes} nodes, {kg.num_edges} edges")

# Scene metadata — the ego vehicle was sideswapped on an urban road / junction
DETECTED_VEHICLE_TYPES = ["car", "van", "truck"]
DETECTED_ROAD_TYPE = "intersection"  # complex junction with roundabout elements
KEYWORD_HINTS = ["sideswipe", "lane change", "blind spot"]

# Retrieve patterns and regulations
retriever = KGRetriever(kg)
scored_patterns = retriever.retrieve_patterns(
    DETECTED_VEHICLE_TYPES, DETECTED_ROAD_TYPE, KEYWORD_HINTS
)
regulations = retriever.retrieve_regulations(DETECTED_ROAD_TYPE, "UK")

print(f"\nRetrieved {len(scored_patterns)} patterns:")
for sp in scored_patterns:
    print(f"  [{sp.relevance_score:.0f}] {sp.pattern_name}")
print(f"Retrieved {len(regulations)} regulations.")

### 6. Compose Prompts (PromptComposer)

In [ ]:
composer = PromptComposer(
    camera_profile=camera_profile,
    scored_patterns=scored_patterns,
    regulations=regulations,
)

system_prompt = composer.build_system_prompt()
task_prompt = composer.build_task_prompt(
    user_query=(
        "The dashcam vehicle itself was sideswapped during this journey. "
        "The collision may not be visually obvious from the front-facing camera — "
        "look for sudden lateral jolts, unexpected stops, or interactions with "
        "adjacent vehicles that indicate side contact with the ego vehicle. "
        "Also check for any vehicle encroaching into the ego vehicle's lane."
    )
)

print(f"System prompt: {len(system_prompt)} chars")
print(f"Task prompt: {len(task_prompt)} chars")
print(f"\n--- System prompt preview ---")
print(system_prompt[:500])


### 7. Build Multimodal Payload & Call LLM (Pass 1)

In [ ]:
# Encode video
logger.info("Encoding video to base64...")
with open(VIDEO_PATH, "rb") as f:
    video_b64 = base64.b64encode(f.read()).decode("utf-8")
video_mime = VIDEO_MIME_TYPES.get(VIDEO_PATH.suffix.lstrip(".").lower(), "video/mp4")

# Build content parts: video + annotated key frames + task prompt
content_parts = [
    {"type": "text", "text": f"[{CAMERA_TYPE}] Video: {VIDEO_PATH.name}"},
    {"type": "image_url", "image_url": {"url": f"data:{video_mime};base64,{video_b64}"}},
]
for kf in key_frames:
    content_parts.append({"type": "text", "text": composer.build_keyframe_annotation(kf.timestamp_seconds)})
    content_parts.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{kf.base64_jpeg}"}})
content_parts.append({"type": "text", "text": task_prompt})

messages_p1 = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": content_parts},
]
print(f"Payload: {len(content_parts)} parts (1 video + {len(key_frames)} frames + prompt)")

In [ ]:
def extract_json_from_response(resp):
    content = resp.choices[0].message.content
    if not content: raise ValueError("Empty response")
    c = content.strip()
    if c.startswith("```"): c = c.split("\n", 1)[1]
    if c.endswith("```"): c = c[:-3].strip()
    return json.loads(c)

logger.info("Pass 1: KG-augmented analysis...")
t0 = time.perf_counter()
raw_p1 = await litellm.acompletion(
    model=MODEL_NAME, messages=messages_p1,
    temperature=0.0, response_format={"type": "json_object"}, cache={"no-store": True},
)
logger.info("Pass 1 done in %.1fs", time.perf_counter() - t0)

parsed_p1 = extract_json_from_response(raw_p1)
print(f"Keys: {list(parsed_p1.keys())}")

# Show reasoning trace
if "reasoning_trace" in parsed_p1:
    for step in ["global_perception", "vehicle_inventory", "interaction_analysis", "causal_conclusion"]:
        print(f"\n-- {step.upper()} --")
        print(parsed_p1["reasoning_trace"].get(step, "N/A")[:500])

p1_crashes = [e for e in parsed_p1.get("events", []) if e.get("event_type") in ("collision", "near_miss")]
print(f"\nPass 1: {len(parsed_p1.get('events',[]))} events, {len(p1_crashes)} crash/near-miss")
for c in p1_crashes:
    print(f"  [{c['event_type']}] {c.get('timestamp_display','?')}: {c.get('description','')[:150]}")

### 8. KG-Grounded Verification (Pass2)

In [ ]:
verification_prompt = composer.build_verification_prompt(parsed_p1)

# Reuse video + key frames, swap in verification prompt
verify_parts = content_parts[:-1] + [{"type": "text", "text": verification_prompt}]
messages_p2 = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": verify_parts},
]

logger.info("Pass 2: Verification...")
t0 = time.perf_counter()
raw_p2 = await litellm.acompletion(
    model=MODEL_NAME, messages=messages_p2,
    temperature=0.0, response_format={"type": "json_object"}, cache={"no-store": True},
)
logger.info("Pass 2 done in %.1fs", time.perf_counter() - t0)

parsed = extract_json_from_response(raw_p2)
p2_crashes = [e for e in parsed.get("events", []) if e.get("event_type") in ("collision", "near_miss")]
print(f"Pass 2: {len(parsed.get('events',[]))} events, {len(p2_crashes)} crash/near-miss")
for c in p2_crashes:
    print(f"  [{c['event_type']}] {c.get('timestamp_display','?')}: {c.get('description','')[:150]}")

if len(p1_crashes) != len(p2_crashes):
    print(f"\n⚠ Pass 2 changed crash count: {len(p1_crashes)} → {len(p2_crashes)}")

### 9. Parse Into Pydantic Models

In [ ]:
# Reasoning trace
reasoning_trace = ReasoningTrace(**parsed["reasoning_trace"]) if "reasoning_trace" in parsed else None

# Environment
environment = EnvironmentContext(**parsed["environment"]) if parsed.get("environment") else None

# Detected objects
detected_objects = [VideoObjectObservation(**o) for o in parsed.get("detected_objects", [])]

# Events (with nested crash detail hydration)
events = []
for evt_raw in parsed.get("events", []):
    evt = dict(evt_raw)
    cd_raw = evt.pop("witnessed_crash_detail", None)
    cd = None
    if cd_raw and isinstance(cd_raw, dict):
        dd = dict(cd_raw)
        parties = [CrashInvolvedParty(**p) for p in dd.pop("involved_parties", [])]
        cd = WitnessedCrashDetail(involved_parties=parties, **dd)
    events.append(SceneEvent(witnessed_crash_detail=cd, **evt))

# Final output
output = VideoSceneAnalysisOutput(
    task_name="video_scene_analysis_kg_integration",
    camera_type=CAMERA_TYPE,
    video_file_name=VIDEO_PATH.name,
    environment=environment,
    detected_objects=detected_objects,
    events=events,
    summary=parsed.get("summary"),
    reasoning_trace=reasoning_trace,
    raw_output=parsed,
)

n_crash = sum(1 for e in events if e.event_type in ("collision", "near_miss"))
print(f"Objects: {len(detected_objects)}, Events: {len(events)} ({n_crash} crash/near-miss)")
print(f"Summary: {output.summary}")

### 10. Explore Result

In [ ]:
# Chain-of-thought reasoning trace
if reasoning_trace:
    print("=" * 60, "\n  REASONING TRACE\n", "=" * 60)
    for field in ["global_perception", "vehicle_inventory", "interaction_analysis", "causal_conclusion"]:
        print(f"\n-- {field.replace('_',' ').upper()} --")
        print(getattr(reasoning_trace, field))

In [ ]:
# Environment
if output.environment:
    e = output.environment
    print("=" * 60, "\n  Environment\n", "=" * 60)
    for f in ["weather", "lighting", "road_type", "road_surface", "traffic_density", "additional_notes"]:
        v = getattr(e, f, None)
        if v: print(f"  {f:<18s} {v}")

In [ ]:
# Detected objects
import pandas as pd
if output.detected_objects:
    df = pd.DataFrame([o.model_dump() for o in output.detected_objects])
    display(df.style.format({"confidence": "{:.2%}"}).set_caption(f"Detected Objects ({len(df)})"))

In [ ]:
# All events
for i, ev in enumerate(output.events, 1):
    print(f"\n{'='*60}\n  Event {i}: {ev.event_type}\n{'='*60}")
    print(f"  Time: {ev.timestamp_display} ({ev.timestamp_seconds:.1f}s)")
    print(f"  Severity: {ev.severity or 'N/A'}, Confidence: {ev.confidence:.2%}")
    print(f"  Description: {ev.description}")
    if ev.witnessed_crash_detail:
        cd = ev.witnessed_crash_detail
        print(f"  Impact: {cd.impact_type}, Cause: {cd.probable_cause}")
        print(f"  Dashcam involved: {cd.dashcam_vehicle_involved}")
        print(f"  Sequence: {cd.sequence_of_events}")
        if cd.fault_indicators:
            print(f"  Fault indicators: {', '.join(cd.fault_indicators)}")
        for j, p in enumerate(cd.involved_parties, 1):
            print(f"    Party {j}: {p.road_user_type} — {p.description}")
            print(f"      Before: {p.action_before_event}")
            print(f"      After:  {p.action_after_event}")
            print(f"      Damage: {p.apparent_damage}")

In [ ]:
# Token usage
for label, resp in [("Pass 1", raw_p1), ("Pass 2", raw_p2)]:
    u = getattr(resp, "usage", None)
    if u:
        print(f"{label}: prompt={getattr(u,'prompt_tokens','?')}, "
              f"completion={getattr(u,'completion_tokens','?')}, "
              f"total={getattr(u,'total_tokens','?')}")

### 11. Export

In [ ]:
# Save structured output
out_path = Path("./scene_analysis_verification_results.json")
out_path.write_text(json.dumps(output.model_dump(), indent=2, default=str))
print(f"Results saved: {out_path.resolve()}")

# Export KG as GraphML
kg.to_graphml("./verification_knowledge_graph.graphml")
print("KG exported: ./verification_knowledge_graph.graphml")

In [ ]:
# Raw JSON for inspection
display(JSON(output.raw_output, root="raw_model_output"))